In [1]:
# Clone the latest project from GitHub
!git clone https://github.com/DTD-Wijesinghe/IT3091---Machine-Learning-Assignment.git
%cd /content/IT3091---Machine-Learning-Assignment

Cloning into 'IT3091---Machine-Learning-Assignment'...
remote: Enumerating objects: 110, done.
remote: Counting objects: 100% (110/110), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 110 (delta 39), reused 24 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (110/110), 7.83 MiB | 4.33 MiB/s, done.
Resolving deltas: 100% (39/39), done.
/content/IT3091---Machine-Learning-Assignment


In [2]:
!pip -q install openpyxl joblib

import os
import re
import json
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

REPO_PATH = Path("/content/IT3091---Machine-Learning-Assignment")
RAW_PATH = REPO_PATH / "Raw Datasets"
PREPROCESSED_PATH = REPO_PATH / "Preprocessed Datasets"

PREPROCESSED_PATH.mkdir(parents=True, exist_ok=True)

print("Raw data:", RAW_PATH)
print("Preprocessed data:", PREPROCESSED_PATH)

Raw data: /content/IT3091---Machine-Learning-Assignment/Raw Datasets
Preprocessed data: /content/IT3091---Machine-Learning-Assignment/Preprocessed Datasets


In [3]:
PRODUCTION_FILE = RAW_PATH / "researchData.xlsx"

if not PRODUCTION_FILE.exists():
    matches = list(RAW_PATH.glob("*researchData*.xlsx"))
    if not matches:
        raise FileNotFoundError("researchData.xlsx was not found in Raw Datasets.")
    PRODUCTION_FILE = matches[0]

print("Production file:", PRODUCTION_FILE.name)

Production file: researchData.xlsx


In [4]:
prod_raw = pd.read_excel(PRODUCTION_FILE)
prod_raw = prod_raw.loc[:, ~prod_raw.columns.astype(str).str.startswith("Unnamed")]
prod_raw = prod_raw.drop(columns=["index"], errors="ignore")

print("Shape:", prod_raw.shape)
display(prod_raw.head())

Shape: (94755, 7)


,District,Season,CropCategory,Crop,Year,Extent,Production
0,National Total,Yala,Cereals,Kurakkan,2001,650.0,422.0
1,National Total,Maha,Cereals,Kurakkan,2001,"4,986.0","3,775.0"
2,National Total,Total,Cereals,Kurakkan,2001,"5,636.0","4,197.0"
3,National Total,Yala,Cereals,Kurakkan,2002,647.0,408.0
4,National Total,Maha,Cereals,Kurakkan,2002,"4,830.0","3,663.0"


In [5]:
prod = prod_raw.copy()

for c in ["District", "Season", "CropCategory", "Crop"]:
    prod[c] = prod[c].astype("string").str.strip()

for c in ["Year", "Extent", "Production"]:
    prod[c] = pd.to_numeric(
        prod[c].astype(str).str.replace(",", "", regex=False).str.strip(),
        errors="coerce"
    )

VEG_CATEGORIES = ["Up Country Vegetable", "Low Country Vegetable"]

prod = prod[
    (prod["District"].eq("National Total")) &
    (prod["Season"].isin(["Maha", "Yala"])) &
    (prod["CropCategory"].isin(VEG_CATEGORIES)) &
    (prod["Year"].between(2012, 2025))
].copy()

prod = prod[
    (prod["Extent"] > 0) &
    prod["Production"].notna() &
    (prod["Production"] >= 0)
].copy()

prod["Year"] = prod["Year"].astype(int)
prod["Yield_t_ha"] = prod["Production"] / prod["Extent"]
prod = prod.replace([np.inf, -np.inf], np.nan).dropna(subset=["Yield_t_ha"])

print("Clean production shape:", prod.shape)
display(prod.head())

Clean production shape: (699, 8)


,District,Season,CropCategory,Crop,Year,Extent,Production,Yield_t_ha
30006,National Total,Yala,Low Country Vegetable,Luffa,2012,1861.0,17053.0,9.163353
30007,National Total,Maha,Low Country Vegetable,Luffa,2012,2840.0,25733.0,9.060915
30009,National Total,Yala,Low Country Vegetable,Luffa,2013,1985.0,17801.0,8.967758
30010,National Total,Maha,Low Country Vegetable,Luffa,2013,3005.0,28598.0,9.516805
30012,National Total,Yala,Low Country Vegetable,Luffa,2014,1881.0,16066.0,8.541201


In [6]:
def iqr_flag_series(s):
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    return (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)

prod["Yield_IQR_Flag"] = (
    prod.groupby("Crop")["Yield_t_ha"]
        .transform(iqr_flag_series)
        .fillna(False)
        .astype(bool)
)

print("Rows flagged for review:", int(prod["Yield_IQR_Flag"].sum()))

Rows flagged for review: 35


In [7]:
production_output = PREPROCESSED_PATH / "01_production_preprocessed.csv"
prod.to_csv(production_output, index=False)

print("Saved:", production_output)
print("Rows:", len(prod))

Saved: /content/IT3091---Machine-Learning-Assignment/Preprocessed Datasets/01_production_preprocessed.csv
Rows: 699


In [ ]:
# Save this executed notebook and push this member's work to GitHub
from getpass import getpass
from google.colab import _message
import subprocess

MEMBER_NAME = 'Wijesinghe.D.T.D'
NOTEBOOK_NAME = '01_IT24100858_Wijesinghe_DTD_Production.ipynb'
FILES_TO_ADD = ['Preprocessed Datasets/01_production_preprocessed.csv']

# Save the current Colab notebook inside the cloned repository
current_notebook = _message.blocking_request("get_ipynb", request="", timeout_sec=10)
notebook_path = REPO_PATH / NOTEBOOK_NAME

with open(notebook_path, "w", encoding="utf-8") as f:
    json.dump(current_notebook["ipynb"], f, ensure_ascii=False, indent=1)

print("Saved notebook:", notebook_path.name)

# Git identity
github_email = input("GitHub email: ").strip()
github_username = input("GitHub username: ").strip()
github_token = getpass("GitHub Personal Access Token: ")

subprocess.run(["git", "config", "user.name", MEMBER_NAME], cwd=REPO_PATH, check=True)
subprocess.run(["git", "config", "user.email", github_email], cwd=REPO_PATH, check=True)

# Add notebook and this member's output files
for item in [NOTEBOOK_NAME] + FILES_TO_ADD:
    subprocess.run(["git", "add", item], cwd=REPO_PATH, check=True)

status = subprocess.run(
    ["git", "status", "--short"],
    cwd=REPO_PATH,
    text=True,
    capture_output=True,
    check=True
)
print(status.stdout)

# Commit only when there is something new
if status.stdout.strip():
    subprocess.run(
        ["git", "commit", "-m", 'Member 1: Add production preprocessing'],
        cwd=REPO_PATH,
        check=True
    )
else:
    print("Nothing new to commit.")

# Secure GitHub authentication for this push
askpass = Path("/tmp/git_askpass.sh")
askpass.write_text(
    '#!/bin/sh\n'
    'case "$1" in\n'
    '  *Username*) echo "$GITHUB_USERNAME" ;;\n'
    '  *Password*) echo "$GITHUB_TOKEN" ;;\n'
    'esac\n'
)
askpass.chmod(0o700)

env = os.environ.copy()
env["GITHUB_USERNAME"] = github_username
env["GITHUB_TOKEN"] = github_token
env["GIT_ASKPASS"] = str(askpass)

result = subprocess.run(
    ["git", "push", "origin", "main"],
    cwd=REPO_PATH,
    env=env,
    text=True,
    capture_output=True
)

print(result.stdout)
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Push failed. Check the GitHub username, token and repository permission.")

print("Push completed.")